# Foundry IQ

## Intro
[Foundry IQ]() provides an intelligent orchestration layer for agents, automatically selecting the right grounding sources for different tasks. Instead of relying on complex prompting strategies or manual tool‑selection logic, 
Foundry IQ analyzes the agent’s intent and identifies most relevant knowledge sources using the agents chat content with instructions or system prompt as well as LLM interactions:

![FoundryIQ-Overview](../assets/img/foundryiq-detail.png)


## Read connection information

The Azure CLI script [setup.azcli](../setup/setup.azcli) can be used to create the necessary Azure environment to execute the notebook code cells. You can also rename [config.env.template](../config/config.env.template) to `config.env`and provide endpoint information, deployment names and access tokens.

In [14]:
#r "nuget: Azure.AI.OpenAI, 2.1.0"
#r "nuget: Azure.Search.Documents, 11.8.0-beta.1"
#r "nuget: DotNetEnv, 3.1.1"

using DotNetEnv;
using System.IO;
using Azure;

string configurationFile = "../config/config.env";
Env.Load(configurationFile);

string openAIApiKey = Env.GetString("OPENAI_APIKEY");
string openAIEndpoint = Env.GetString("OPENAI_ENDPOINT");
string openAIChatDeployment = Env.GetString("OPENAI_CHATDEPLOYMENT");
string openAIChatModel = Env.GetString("OPENAI_CHATMODEL");
string openAIEmbeddingDeployment = Env.GetString("OPENAI_EMBEDDINGDEPLOYMENT");
string openAIEmbeddingModel = Env.GetString("OPENAI_EMBEDDINGMODEL");

string aiSearchEndpoint = Env.GetString("AISEARCH_ENDPOINT");
string aiSearchApiKey = Env.GetString("AISEARCH_APIKEY");

Console.WriteLine("Configuration loaded...");

Installed Packages Azure.AI.OpenAI, 2.1.0 Azure.Search.Documents, 11.8.0-beta.1 DotNetEnv, 3.1.1

Configuration loaded...


## Define Search Index

The code cell creates the necessary:
- Azure AI Search indexes
- Azure AI Search Vectorizer
- Azure AI Algorythm Configuration
- Azure AI Semantic Configuration


In [15]:
// Define search index fields
using Azure;
using Azure.Search.Documents.Indexes;
using Azure.Search.Documents.Indexes.Models;

string searchIndexNameEventResult = "sportkiosk-results-index";
string searchIndexNameEventDescription = "sportkiosk-description-index";
string vectorizerName = "azure_openai_text_3_large";
string vectorSearchProfileName = "hnsw_text_3_large";
string algorithmConfigurationName = "alg";
string semanticConfigName = "semantic_config";

AzureKeyCredential searchKeyCredential = new AzureKeyCredential(aiSearchApiKey);
AzureKeyCredential openAIKeyCredential = new AzureKeyCredential(openAIApiKey);

        
List<SearchField> searchFields = new List<SearchField>
{
    new SimpleField("id", SearchFieldDataType.String) { 
        IsKey = true, 
        IsFilterable = true, 
        IsSortable = true, 
        IsFacetable = true 
    },
    new SearchField("page_chunk", SearchFieldDataType.String) { 
        IsFilterable = false, 
        IsSortable = false, 
        IsFacetable = false 
    },
    new SearchField("page_embedding_text_3_large", SearchFieldDataType.Collection(SearchFieldDataType.Single)) { 
        VectorSearchDimensions = 3072, 
        VectorSearchProfileName = vectorSearchProfileName 
    }
};

Console.WriteLine("Search fields defined...");

// Define a vectorizer
AzureOpenAIVectorizer azureOpenAIVectorizer = new AzureOpenAIVectorizer(vectorizerName)
{
    Parameters = new AzureOpenAIVectorizerParameters
    {
        ResourceUri = new Uri(openAIEndpoint),
        DeploymentName = openAIEmbeddingDeployment,
        ModelName = openAIEmbeddingModel,
        ApiKey = openAIApiKey,
    }
};

// Define a vector search profile and algorithm
VectorSearch vectorSearch = new VectorSearch()
{
    Profiles = {
        new VectorSearchProfile(vectorSearchProfileName, algorithmConfigurationName) {
            VectorizerName = vectorizerName
        }
    },
    Algorithms ={
        new HnswAlgorithmConfiguration(algorithmConfigurationName)
    },
    Vectorizers ={
        azureOpenAIVectorizer
    }
};

// Define a semantic configuration
SemanticConfiguration semanticConfiguration = new SemanticConfiguration(semanticConfigName,
    prioritizedFields: new SemanticPrioritizedFields
    {
        ContentFields = { new SemanticField("page_chunk") }
    }
);

SemanticSearch semanticSearch = new SemanticSearch()
{
    DefaultConfigurationName = semanticConfigName,
    Configurations = { semanticConfiguration }
};

Console.WriteLine("Vector search and semantic search configurations defined...");

// Create indexes
SearchIndex searchIndexEventResult = new SearchIndex(searchIndexNameEventResult)
{
    Fields = searchFields,
    VectorSearch = vectorSearch,
    SemanticSearch = semanticSearch
};

SearchIndexClient searchIndexClient = new SearchIndexClient(
    new Uri(aiSearchEndpoint), 
    searchKeyCredential
);

await searchIndexClient.CreateOrUpdateIndexAsync(searchIndexEventResult);
Console.WriteLine($"Index '{searchIndexNameEventResult}' created or updated successfully...");

SearchIndex searchIndexEventDescription = new SearchIndex(searchIndexNameEventDescription)
{
    Fields = searchFields,
    VectorSearch = vectorSearch,
    SemanticSearch = semanticSearch
};

await searchIndexClient.CreateOrUpdateIndexAsync(searchIndexEventDescription);
Console.WriteLine($"Index '{searchIndexNameEventDescription}' created or updated successfully...");


Search fields defined...
Vector search and semantic search configurations defined...
Index 'sportkiosk-results-index' created or updated successfully...
Index 'sportkiosk-description-index' created or updated successfully...


## Create Embeddings

Textual information from: 

- [sporteventresult.md](../assets/sporteventresult.md)
- [sporteventwinner.md](../assets/sporteventwinner.md)
- [supersportchapionship.md](../assets/supersportchampionship.md)

is embedded and stored in the corresponding search indexes.

In [16]:
using Azure.AI.OpenAI;
using OpenAI.Embeddings;
using System.ClientModel;
using Azure.Search.Documents;

//Define knowledge documents
public class KnowledgeDocument
{
    public string Id { get; set; }
    public string Page_Chunk { get; set; }
    public float[] Page_Embedding_Text_3_Large { get; set; }
}   

// Read grounding files
string groundingSportEventWinnerFile = "../assets/sporteventwinner.md";
string groundingSportEventResultFile = "../assets/sporteventresult.md";
string groundingSuperSportChampionShipFile = "../assets/supersportchampionship.md";

string[] groundingSportEventWinner = await File.ReadAllLinesAsync(groundingSportEventWinnerFile);
string[] groundingSportEventResult = await File.ReadAllLinesAsync(groundingSportEventResultFile);
string[] groundingSuperSportChampionShip = await File.ReadAllLinesAsync(groundingSuperSportChampionShipFile);


// Create the embeddings client
EmbeddingClient embeddingClient = new AzureOpenAIClient(new Uri(openAIEndpoint), openAIKeyCredential)
    .GetEmbeddingClient(openAIEmbeddingDeployment);

ClientResult<OpenAIEmbeddingCollection> openAIEmbeddingCollection = await embeddingClient.GenerateEmbeddingsAsync(
    new List<string> { 
        groundingSportEventResult[0], 
        groundingSportEventWinner[0],
        string.Join("\n", groundingSuperSportChampionShip)
    }
);

// Create knowledge documents and upload to search index
List<KnowledgeDocument> knowledgeDocuments = new List<KnowledgeDocument>();

string pageChunk = string.Join("\n", groundingSportEventResult.Skip(1));
knowledgeDocuments.Add(new KnowledgeDocument
{
    Id = groundingSportEventResultFile.Replace("/","").Replace(":","").Replace("..","").Replace(".","_"),
    Page_Chunk = pageChunk,
    Page_Embedding_Text_3_Large = openAIEmbeddingCollection.Value[0].ToFloats().ToArray(),
});
knowledgeDocuments.Add(new KnowledgeDocument
{
    Id = groundingSportEventWinnerFile.Replace("/","").Replace(":","").Replace("..","").Replace(".","_"),
    Page_Chunk = string.Join("\n", groundingSportEventWinner.Skip(1)),
    Page_Embedding_Text_3_Large = openAIEmbeddingCollection.Value[1].ToFloats().ToArray(),
});

// Upload knowledge documents to search index
SearchClient searchClient = new SearchClient(
    new Uri(aiSearchEndpoint), 
    searchIndexNameEventResult, 
    searchKeyCredential
);
await searchClient.UploadDocumentsAsync(knowledgeDocuments);

knowledgeDocuments.Clear();
knowledgeDocuments.Add(new KnowledgeDocument
{
    Id = groundingSportEventWinnerFile.Replace("/","").Replace(":","").Replace("..","").Replace(".","_"),
    Page_Chunk = string.Join("\n", groundingSuperSportChampionShip),
    Page_Embedding_Text_3_Large = openAIEmbeddingCollection.Value[1].ToFloats().ToArray(),
});
searchClient = new SearchClient(
    new Uri(aiSearchEndpoint), 
    searchIndexNameEventDescription, 
    searchKeyCredential
);
await searchClient.UploadDocumentsAsync(knowledgeDocuments);

Console.WriteLine("Knowledge documents uploaded successfully...");
    

Knowledge documents uploaded successfully...


## Create Knowledge Sources

Two knowledge sources:

- sportkiosk-knowledgesoruce-result
- sportkiosk-knowledgesource-description

are created backed by the previously created Azure AI Search indexes.

In [17]:
// Create a knowledge source

string knowledgeSourceNameResult = "sportkiosk-knowledgesource-result";
string knowledgeSourceNameDescription = "sportkiosk-knowledgesource-description";

SearchIndexKnowledgeSource searchIndexKnowledgeSource = new SearchIndexKnowledgeSource(
    knowledgeSourceNameResult,
    new SearchIndexKnowledgeSourceParameters(searchIndexName: searchIndexNameEventResult)
    {
        SourceDataFields = { new SearchIndexFieldReference(name: "id"), new SearchIndexFieldReference(name: "page_chunk") }
    }
);
await searchIndexClient.CreateOrUpdateKnowledgeSourceAsync(searchIndexKnowledgeSource);

searchIndexKnowledgeSource = new SearchIndexKnowledgeSource(
    knowledgeSourceNameDescription,
    new SearchIndexKnowledgeSourceParameters(searchIndexName: searchIndexNameEventDescription)
    {
        SourceDataFields = { new SearchIndexFieldReference(name: "id"), new SearchIndexFieldReference(name: "page_chunk") }
    }
);
await searchIndexClient.CreateOrUpdateKnowledgeSourceAsync(searchIndexKnowledgeSource);

Console.WriteLine($"Knowledge source '{knowledgeSourceNameResult}' created or updated successfully...");
Console.WriteLine($"Knowledge source '{knowledgeSourceNameDescription}' created or updated successfully...");


Knowledge source 'sportkiosk-knowledgesource-result' created or updated successfully...
Knowledge source 'sportkiosk-knowledgesource-description' created or updated successfully...


## Create Knowledge Base

A knowledge base named `sportkiosk-knowledgebase` is created which is backed by the two previously created knowledge sources.

In [ ]:
using Azure.Search.Documents.KnowledgeBases.Models;

// Create a knowledge base

string knowledgeBaseName = "sportkiosk-knowledgebase";

AzureOpenAIVectorizerParameters openAiParameters = new AzureOpenAIVectorizerParameters()
{
    ResourceUri = new Uri(openAIEndpoint),
    DeploymentName = openAIChatDeployment,
    ModelName = openAIChatModel,
    ApiKey = openAIApiKey,
};

KnowledgeBaseAzureOpenAIModel knowledgeBaseAzureOpenAIModel = new KnowledgeBaseAzureOpenAIModel(azureOpenAIParameters: openAiParameters);

KnowledgeBase knowledgeBase = new KnowledgeBase(
    knowledgeBaseName,
    new KnowledgeSourceReference[] { 
        new KnowledgeSourceReference(knowledgeSourceNameResult),
        new KnowledgeSourceReference(knowledgeSourceNameDescription) 
    }
);
knowledgeBase.RetrievalReasoningEffort = new KnowledgeRetrievalLowReasoningEffort();
knowledgeBase.AnswerInstructions = "Provide a short answer based on the retrieved documents.";
knowledgeBase.Models.Add(knowledgeBaseAzureOpenAIModel);
await searchIndexClient.CreateOrUpdateKnowledgeBaseAsync(knowledgeBase);

Console.WriteLine($"Knowledge base '{knowledgeBaseName}' created or updated successfully...");


Knowledge base 'sportkios-knowledgebase' created or updated successfully...


## Get grounding

For the fictious agent task:

    "Who won the Super Sport Championship 2025? Just retrieve the winner and no additional information?"

the previously created knowledge source is used.

In [ ]:
using Azure.Search.Documents.KnowledgeBases;
using Azure.Search.Documents.KnowledgeBases.Models;
using System.Text.Json;

string prompt = "Who won the Super Sport Championship 2025? Just retrieve the winner and no additional information?";

// Get grounding
KnowledgeBaseRetrievalClient knowledgeBaseRetrievalClient = new KnowledgeBaseRetrievalClient(
    new Uri(aiSearchEndpoint),
    knowledgeBaseName,
    searchKeyCredential
);

// Retrieve grounding, activity, and references
KnowledgeBaseRetrievalRequest knowledgeBaseRetrievalRequest = new KnowledgeBaseRetrievalRequest();
KnowledgeBaseMessageTextContent knowledgeBaseMessageContent = new KnowledgeBaseMessageTextContent(prompt);
KnowledgeBaseMessage knowledgeBaseMessage = new KnowledgeBaseMessage(
    new KnowledgeBaseMessageContent[] { 
        knowledgeBaseMessageContent 
    }
); 
knowledgeBaseMessage.Role = "user";
knowledgeBaseRetrievalRequest.Messages.Add(knowledgeBaseMessage);
knowledgeBaseRetrievalRequest.RetrievalReasoningEffort = new KnowledgeRetrievalLowReasoningEffort();
Response<KnowledgeBaseRetrievalResponse> knowledgeBaseRetrievalResponse = 
    await knowledgeBaseRetrievalClient.RetrieveAsync(knowledgeBaseRetrievalRequest);

Console.WriteLine($"Query: {prompt}");
Console.WriteLine("Grounding:");
Console.WriteLine((knowledgeBaseRetrievalResponse.Value.Response[0].Content[0] as KnowledgeBaseMessageTextContent)!.Text);


Query: Who won the Super Sport Championship 2025? Just retrieve the winner and no additional information.
Grounding:
[{"ref_id":0,"content":"\n| Year | Club A                          | Club B                          |\n|------|---------------------------------|---------------------------------|\n| 2016 | New York Charging Bulls         | Tokyo Soaring Eagles            |\n| 2017 | London Roaring Lions            | Shanghai Sprinting Tigers       |\n| 2018 | Paris Leaping Panthers          | Berlin Striking Bears           |\n| 2019 | Madrid Running Stallions        | Seoul Gliding Hawks             |\n| 2020 | Rome Rising Phoenixes           | Chicago Crushing Crocodiles     |\n| 2021 | Dublin Sprinting Horses         | Los Angeles Racing Sharks       |\n| 2022 | Barcelona Jumping Jaguars       | Hong Kong Climbing Dragons      |\n| 2023 | Amsterdam Howling Wolves        | San Francisco Pouncing Leopards |\n| 2024 | Vienna Charging Rhinos          | Singapore Diving Falcons        |\

Because the query is winners of the Super Sport Championship, information from the knowledge source  `sportkiosk-knowledgesource-result` is retrieved. The knowledge source points to the search index which contains information from the [../assets/sporteventresult.md](../assets/sporteventwinner.md) and [../assets/sporteventwinner.md](../assets/sporteventwinner.md) files.

## Get Grounding

For the fictious agent task:

    "What is the Super Sport Championship? Provide a breif description of the event?"

the previously created knowledge source is used.


In [ ]:

prompt = "What is the Super Sport Championship? Provide a breif description of the event?";

knowledgeBaseRetrievalRequest.Messages.Clear();

knowledgeBaseMessageContent = new KnowledgeBaseMessageTextContent(prompt);
KnowledgeBaseMessage knowledgeBaseMessage = new KnowledgeBaseMessage(
    new KnowledgeBaseMessageContent[] { 
        knowledgeBaseMessageContent 
    }
); 

knowledgeBaseMessage.Role = "user";
knowledgeBaseRetrievalRequest.Messages.Add(knowledgeBaseMessage);
knowledgeBaseRetrievalRequest.RetrievalReasoningEffort = new KnowledgeRetrievalLowReasoningEffort();
Response<KnowledgeBaseRetrievalResponse> knowledgeBaseRetrievalResponse = 
    await knowledgeBaseRetrievalClient.RetrieveAsync(knowledgeBaseRetrievalRequest);


Console.WriteLine($"Query: {prompt}");
Console.WriteLine("Grounding:");
Console.WriteLine((knowledgeBaseRetrievalResponse.Value.Response[0].Content[0] as KnowledgeBaseMessageTextContent)!.Text);


Query: What is the Super Sport Championship? Provide a breif description of the event.
Grounding:
[{"ref_id":0,"content":"# Super Sport Champion Ship\n## About\nThe Super Sport Championship is a unique annual competition that brings together scientists from across the globe. Participants gather to test their knowledge and problem-solving skills by answering a series of challenging scientific questions. The event fosters international collaboration and friendly rivalry, as teams strive to demonstrate their expertise in various scientific disciplines.\nEach correct answer in the competition earns a point for the team, while incorrect responses result in a deduction from the team’s total score. The contest continues until either 120 minutes have elapsed or a maximum of 100 questions have been answered, whichever comes first. This structure ensures a dynamic and fast-paced environment, encouraging strategic thinking and teamwork throughout the event.\n\n\n"}]


Because the query is about explaining the fictious Super Sport Championship information from the knowledge source  `sportkiosk-knowledgesource-description` is retrieved. The knowledge source points to the search index which contains information from the [../assets/supersportchampionship.md](../assets/supersportchampionship.md) file